# 08 - Final Comparison & Trade-off Discussion

Assignment Task 6. Compare the custom resource-constrained CNN (**Model B**, under 100,000 trainable parameters) against the two fine-tuned lightweight SOTA models, and discuss the trade-offs of building a sub-100k model from scratch versus using an industry-standard pretrained one, focusing on **accuracy, memory footprint, and computational cost**.

Model A is carried through the tables as the standard-convolution baseline, but the discussion in Section 10 centres on Model B versus the SOTA models, which is what Task 6 asks for. Model A versus Model B is covered in `05_custom_models_evaluation.ipynb` (Task 4).

**Expects** `results/metrics/{model_a,model_b,mobilenet,efficientnet}.json` to exist, i.e. notebooks 02, 03, 06 and 07 have all been run.

In [ ]:
# --- Environment setup (Google Colab + shared Google Drive, with a local fallback) ---
import os

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Mount the team's shared Drive - this is where notebooks 02/03/06/07 saved
    # their metrics JSONs.
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/EN3150-A03-Resource-Constrained-CNN"
else:
    # Not in Colab - use this repo's own local folders instead of Drive.
    REPO_DIR = os.path.abspath("..")

METRICS_DIR = os.path.join(REPO_DIR, "results", "metrics")
TABLES_DIR = os.path.join(REPO_DIR, "results", "tables")
FIGURES_FINAL_DIR = os.path.join(REPO_DIR, "results", "figures", "final")

for d in (TABLES_DIR, FIGURES_FINAL_DIR):
    os.makedirs(d, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("REPO_DIR:", REPO_DIR)

## 1. Imports

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import keras
from keras import layers

print("Keras:", keras.__version__)

## 2. Load All Saved Metrics

In [ ]:
def load_metrics(name):
    with open(os.path.join(METRICS_DIR, f"{name}.json")) as f:
        return json.load(f)


metrics_a = load_metrics("model_a")
metrics_b = load_metrics("model_b")
metrics_mobilenet = load_metrics("mobilenet")
metrics_efficientnet = load_metrics("efficientnet")

for m in (metrics_a, metrics_b, metrics_mobilenet, metrics_efficientnet):
    print(f"{m['model_name']:<28} test accuracy {m['test_accuracy']:.4f}")

## 3. Putting the Four Notebooks on a Common Footing

The four notebooks were written by different group members and their JSON files do not share one schema, so a few fields have to be reconciled before they can go in one table.

**Model size.** Each notebook reports the size of its saved `.keras` file, but that archive also stores the optimizer state (Adam keeps two moment estimates per trainable parameter), so it is roughly three times what would actually be deployed. Model A's 1.44M parameters need 5.6 MB of float32 weights but its checkpoint is 16.6 MB. For a memory-footprint comparison we therefore derive a **weights-only float32 size** as `total_params * 4 / 1024` KB, uniformly for all four models. Where a notebook already stored that figure (Model B, EfficientNet-B0), the derived value reproduces it exactly, which is the check that the derivation is right. Both columns are kept: the archive size is what Task 4's table asks for, the weights-only size is what the Task 6 memory discussion needs.

**MAC counts.** Only notebooks 03 and 07 recorded them. Section 4 fills in the other two.

**Input resolution.** Model A, Model B and EfficientNet-B0 all run on the 64x64 arrays from notebook 01. MobileNetV2 upsamples them to 96x96 in its input pipeline, because its five stride-2 stages leave too small a feature map at 64x64. The data splits are identical in every case, but MobileNetV2's accuracy is not measured at the same input resolution as the others, and Section 10 returns to what that means.

In [ ]:
# Which runtime each notebook actually trained on, read from the "GPU available"
# line each one printed. Notebooks 02 and 06 had a GPU; 03 and 07 did not.
# This is why the per-epoch times must not be compared directly (Section 9).
RUNTIME = {
    "Model A (Standard CNN)": "GPU",
    "Model B (Lightweight CNN)": "CPU",
    "MobileNetV2": "GPU",
    "EfficientNet-B0": "CPU",
}


def summarize(metrics, input_size, macs=None, trainable=None):
    """One row of the comparison, with the fields the notebooks disagree on
    derived consistently. `trainable` left as None falls back to the stored
    value, or to NaN where the notebook never recorded one - never to the
    total, which would silently overstate it."""
    total_params = metrics["total_params"]
    return {
        "Model": metrics["model_name"],
        "Input": f"{input_size}x{input_size}",
        "Total Parameters": total_params,
        "Trainable Parameters": trainable if trainable is not None else metrics.get("trainable_params", np.nan),
        "Weights fp32 (KB)": total_params * 4 / 1024,
        "Checkpoint on Disk (KB)": metrics.get("model_size_kb", metrics.get("model_size_mb", 0) * 1024),
        "MACs/image": macs if macs is not None else metrics.get("macs"),
        "Test Accuracy": metrics["test_accuracy"],
        "Precision (macro)": metrics["test_precision_macro"],
        "Recall (macro)": metrics["test_recall_macro"],
        "Avg Time/Epoch (s)": metrics["avg_time_per_epoch_sec"],
        "Runtime": RUNTIME[metrics["model_name"]],
    }


# Check that deriving the weights-only size from total_params reproduces the
# figure the notebooks that stored it already computed.
for m in (metrics_b, metrics_efficientnet):
    derived = m["total_params"] * 4 / 1024
    assert abs(derived - m["weights_size_kb"]) < 0.01, m["model_name"]
print("Weights-only size derivation matches the stored values.")

## 4. Filling in the Missing MAC Counts

Task 6 asks for computational cost, and MACs are the hardware-independent way to state it: one MAC is one multiply plus one accumulate, and unlike wall-clock time the count does not change with the runtime a notebook happened to get.

Notebook 03 recorded Model B's and notebook 07 recorded EfficientNet-B0's. The two missing ones are computed below by rebuilding the architectures. `weights=None` is used for MobileNetV2 because only the layer shapes matter for a MAC count, not the trained values, so nothing is downloaded.

In [ ]:
def iter_leaf_layers(layer):
    """Yield every non-Model layer, descending into nested models."""
    if isinstance(layer, keras.Model):
        for child in layer.layers:
            yield from iter_leaf_layers(child)
    else:
        yield layer


def count_macs(model):
    """Multiply-accumulate operations for one forward pass over one image."""
    total = 0
    for layer in iter_leaf_layers(model):
        if isinstance(layer, layers.SeparableConv2D):
            positions = int(np.prod(layer.output.shape[1:-1]))
            total += positions * (int(np.prod(layer.depthwise_kernel.shape))
                                  + int(np.prod(layer.pointwise_kernel.shape)))
        elif isinstance(layer, (layers.DepthwiseConv2D, layers.Conv2D)):
            positions = int(np.prod(layer.output.shape[1:-1]))
            total += positions * int(np.prod(layer.kernel.shape))
        elif isinstance(layer, layers.Dense):
            total += int(np.prod(layer.kernel.shape))
    return total


# --- Model A, rebuilt from 02_model_A_standard_cnn.ipynb ---
def conv_block(filters):
    return [
        layers.Conv2D(filters, kernel_size=3, padding="same", use_bias=False),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D(pool_size=2),
    ]


model_a_layers = [
    layers.Input(shape=(64, 64, 3)),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
]
for filters in (32, 64, 128, 256):
    model_a_layers += conv_block(filters)
model_a_layers += [
    layers.Flatten(),
    layers.Dense(256, use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.5),
    layers.Dense(6, activation="softmax"),
]
model_a_arch = keras.Sequential(model_a_layers)

# --- MobileNetV2 head, rebuilt from 06_sota_mobilenet.ipynb ---
base = keras.applications.MobileNetV2(input_shape=(96, 96, 3), include_top=False, weights=None)
inputs = keras.Input(shape=(96, 96, 3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
mobilenet_arch = keras.Model(inputs, layers.Dense(6, activation="softmax")(x))

# The rebuilds must reproduce the parameter counts the original notebooks saved,
# otherwise the architecture transcribed here is not the one that was trained.
assert model_a_arch.count_params() == metrics_a["total_params"], "Model A rebuild does not match"
assert mobilenet_arch.count_params() == metrics_mobilenet["total_params"], "MobileNetV2 rebuild does not match"

macs_a = count_macs(model_a_arch)
macs_mobilenet = count_macs(mobilenet_arch)

# Notebook 02 stored only a total, so take Model A's trainable count from the
# rebuild. MobileNetV2's phase-2 trainable count depends on which backbone
# layers notebook 06 unfroze, which it did not record, so it stays blank.
trainable_a = sum(int(np.prod(w.shape)) for w in model_a_arch.trainable_weights)
print()
print(f"Model A trainable parameters (from rebuild): {trainable_a:,}")

print(f"Model A      {macs_a:>12,} MACs/image  (rebuilt)")
print(f"Model B      {metrics_b['macs']:>12,} MACs/image  (from notebook 03)")
print(f"MobileNetV2  {macs_mobilenet:>12,} MACs/image  (rebuilt)")
print(f"EfficientNet {metrics_efficientnet['macs']:>12,} MACs/image  (from notebook 07)")

## 5. Final Comparison Table

Required columns: model name, total parameter count, model size, test accuracy, precision, recall, and training/inference cost.

In [ ]:
comparison_df = pd.DataFrame([
    summarize(metrics_a, 64, macs_a, trainable=trainable_a),
    summarize(metrics_b, 64),
    summarize(metrics_mobilenet, 96, macs_mobilenet),
    summarize(metrics_efficientnet, 64),
])

final_table_path = os.path.join(TABLES_DIR, "final_comparison.csv")
comparison_df.to_csv(final_table_path, index=False)
print("Saved:", final_table_path)

comparison_df.style.format({
    "Total Parameters": "{:,}",
    "Trainable Parameters": "{:,.0f}",
    "Weights fp32 (KB)": "{:,.1f}",
    "Checkpoint on Disk (KB)": "{:,.1f}",
    "MACs/image": "{:,}",
    "Test Accuracy": "{:.4f}",
    "Precision (macro)": "{:.4f}",
    "Recall (macro)": "{:.4f}",
    "Avg Time/Epoch (s)": "{:.1f}",
})

### The same table expressed relative to Model B

Task 6 is a question about what the extra resources buy, so it is easier to read as multiples of the sub-100k model.

In [ ]:
baseline = comparison_df[comparison_df["Model"] == metrics_b["model_name"]].iloc[0]

relative_df = pd.DataFrame({
    "Model": comparison_df["Model"],
    "Parameters vs Model B": comparison_df["Total Parameters"] / baseline["Total Parameters"],
    "Memory vs Model B": comparison_df["Weights fp32 (KB)"] / baseline["Weights fp32 (KB)"],
    "MACs vs Model B": comparison_df["MACs/image"] / baseline["MACs/image"],
    "Accuracy gain (pp)": (comparison_df["Test Accuracy"] - baseline["Test Accuracy"]) * 100,
})

relative_path = os.path.join(TABLES_DIR, "final_comparison_relative.csv")
relative_df.to_csv(relative_path, index=False)
print("Saved:", relative_path)

relative_df.style.format({
    "Parameters vs Model B": "{:.1f}x",
    "Memory vs Model B": "{:.1f}x",
    "MACs vs Model B": "{:.1f}x",
    "Accuracy gain (pp)": "{:+.2f}",
})

## 6. Accuracy vs Memory Footprint and Computational Cost

The two scatter plots below are the core evidence for Task 6: what accuracy each model buys for its memory and its compute. A log scale is used on the x axes because the models span nearly two orders of magnitude.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ["tab:blue", "tab:green", "tab:orange", "tab:red"]

for ax, xcol, xlabel in [
    (axes[0], "Weights fp32 (KB)", "Memory footprint: float32 weights (KB, log scale)"),
    (axes[1], "MACs/image", "Computational cost: MACs per image (log scale)"),
]:
    for (_, row), color in zip(comparison_df.iterrows(), colors):
        ax.scatter(row[xcol], row["Test Accuracy"], s=140, color=color, label=row["Model"], zorder=3)
        ax.annotate(row["Model"], (row[xcol], row["Test Accuracy"]),
                    textcoords="offset points", xytext=(0, 12), ha="center", fontsize=9)
    ax.set_xscale("log")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Test accuracy")
    ax.set_ylim(0.78, 0.95)
    ax.grid(alpha=0.3, zorder=0)

plt.suptitle("Accuracy against memory and compute (all four models, identical test split)")
plt.tight_layout()
tradeoff_path = os.path.join(FIGURES_FINAL_DIR, "accuracy_vs_cost.png")
plt.savefig(tradeoff_path, dpi=150)
plt.show()
print("Saved:", tradeoff_path)

## 7. Efficiency: Accuracy per Unit of Cost

Accuracy alone favours the largest model by construction. On a device with a fixed memory budget the useful question is how much accuracy each kilobyte buys, which is where a sub-100k model is supposed to win.

In [ ]:
efficiency_df = comparison_df.copy()
efficiency_df["Accuracy per MB"] = efficiency_df["Test Accuracy"] / (efficiency_df["Weights fp32 (KB)"] / 1024)
efficiency_df["Accuracy per MMAC"] = efficiency_df["Test Accuracy"] / (efficiency_df["MACs/image"] / 1e6)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, col, title in [
    (axes[0], "Accuracy per MB", "Test accuracy per MB of weights"),
    (axes[1], "Accuracy per MMAC", "Test accuracy per million MACs"),
]:
    ax.bar(efficiency_df["Model"], efficiency_df[col], color=colors)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=20)
    ax.grid(axis="y", alpha=0.3)
    for i, v in enumerate(efficiency_df[col]):
        ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
efficiency_path = os.path.join(FIGURES_FINAL_DIR, "efficiency.png")
plt.savefig(efficiency_path, dpi=150)
plt.show()
print("Saved:", efficiency_path)

efficiency_df[["Model", "Accuracy per MB", "Accuracy per MMAC"]]

## 8. Confusion Matrices, All Four Models

In [ ]:
all_metrics = [metrics_a, metrics_b, metrics_mobilenet, metrics_efficientnet]

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for ax, metrics in zip(axes.ravel(), all_metrics):
    cm = np.array(metrics["confusion_matrix"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=metrics["class_names"], yticklabels=metrics["class_names"], ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{metrics['model_name']}  (acc {metrics['test_accuracy']:.4f})")

plt.tight_layout()
cm_grid_path = os.path.join(FIGURES_FINAL_DIR, "confusion_matrices_all_models.png")
plt.savefig(cm_grid_path, dpi=150)
plt.show()
print("Saved:", cm_grid_path)

In [ ]:
# Where does each model actually lose accuracy? Per-class recall makes the
# confusable pairs explicit and is easier to read than four 6x6 grids.
recall_rows = {}
for metrics in all_metrics:
    cm = np.array(metrics["confusion_matrix"])
    recall_rows[metrics["model_name"]] = cm.diagonal() / cm.sum(axis=1)

per_class_recall = pd.DataFrame(recall_rows, index=metrics_b["class_names"]).T

fig, ax = plt.subplots(figsize=(9, 3.5))
sns.heatmap(per_class_recall, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.6, vmax=1.0, ax=ax)
ax.set_title("Per-class recall (test set)")
plt.tight_layout()
recall_path = os.path.join(FIGURES_FINAL_DIR, "per_class_recall.png")
plt.savefig(recall_path, dpi=150)
plt.show()
print("Saved:", recall_path)

per_class_recall.round(3)

## 9. A Note on Training Time

The `Avg Time/Epoch (s)` column in Section 5 **cannot be compared across rows**, and the report should say so rather than quietly presenting it as a result.

Notebooks 02 and 06 ran on a Colab GPU runtime; notebooks 03 and 07 ran on CPU only, which each notebook's own `GPU available:` line records. Model B therefore appears roughly fourteen times slower per epoch than Model A while performing 5.3 times fewer MACs, which is an artefact of the hardware, not of the architecture. Depthwise separable convolutions are also memory-bandwidth bound rather than compute bound, so even on identical hardware they convert their MAC savings into wall-clock savings less efficiently than the MAC ratio suggests.

MACs per image, used throughout Sections 5 to 7, is the hardware-independent measure and is what Task 6's computational-cost discussion rests on. The per-epoch times are reproduced below with their runtime attached, so the comparison that is valid (same runtime) can still be read off.

In [ ]:
timing_df = comparison_df[["Model", "Runtime", "Avg Time/Epoch (s)", "MACs/image"]].copy()
timing_df = timing_df.sort_values("Runtime")

print("Comparable only within a runtime group:")
print()
for runtime, group in timing_df.groupby("Runtime"):
    print(f"  {runtime}:")
    for _, row in group.iterrows():
        print(f"    {row['Model']:<28} {row['Avg Time/Epoch (s)']:>7.1f} s/epoch"
              f"   {row['MACs/image']:>12,} MACs")
    print()

## 10. Discussion: Custom Sub-100k Model vs Pretrained Lightweight SOTA

**What the extra capacity buys.** Model B reaches 83.18% test accuracy with 66,182 trainable parameters and 11.5M MACs per image. MobileNetV2 reaches 90.38% and EfficientNet-B0 89.24%, so the pretrained models are worth roughly 6 to 7 percentage points. That gain is not free: MobileNetV2 carries 33.6 times Model B's parameters and EfficientNet-B0 60.2 times. In absolute terms Model B's weights occupy 263 KB in float32 against MobileNetV2's 8.6 MB and EfficientNet-B0's 15.5 MB. The accuracy axis compresses into a narrow band while the memory axis spans two orders of magnitude, which is exactly the shape of the trade-off the assignment is asking about.

**Where the accuracy actually comes from.** The SOTA advantage is transfer learning, not architectural sophistication. Both backbones were pretrained on ImageNet, so they arrive with edge, texture and object-part detectors already learned from over a million images, and fine-tuning only has to adapt them to six scene classes and about 12,000 training images. Model B learns everything from scratch on those 12,000 images alone. The comparison is therefore not really 66k parameters against 2.3M parameters, it is 12k images against 1.2M images of prior visual experience. Note also that MobileNetV2 is measured at 96x96 rather than 64x64, because its five stride-2 stages leave too small a feature map at 64x64. Some of its 1.1 point lead over EfficientNet-B0 is that extra input resolution rather than the architecture, so it is not a clean architecture-to-architecture result.

**Compute does not track memory.** EfficientNet-B0 has 1.8 times MobileNetV2's parameters but only 32.0M MACs against 55.0M, because it runs at 64x64 while MobileNetV2 runs at 96x96. MAC count scales with the square of the input resolution and only linearly with width, so on this dataset input size dominates. For a deployment limited by compute rather than by storage, EfficientNet-B0 is the cheaper of the two pretrained models despite being the larger one. This is a useful reminder that "memory footprint" and "computational cost" are separate axes and a model can be favourable on one and not the other.

**Why the custom model still wins on efficiency.** Per megabyte of weights, Model B delivers 3.24 accuracy units against MobileNetV2's 0.105 and EfficientNet-B0's 0.058, factors of 31 and 56. Per million MACs it delivers 0.073 against 0.016 and 0.028. If the target is a microcontroller with a few hundred kilobytes of flash, neither pretrained model fits at all and the comparison ends there: 83.18% from a model that fits is worth more than 90.38% from one that does not. Model B also quantizes cleanly to int8, which would bring it to roughly 66 KB, still with headroom.

**Limitations of the custom model.** It is 7 points behind on accuracy, and the per-class recall figure shows the shortfall is not spread evenly. Glacier is the hardest class for every model here, Model B included at 0.734 recall, but that is not where Model B loses to MobileNetV2. The gap is widest on street, 15.9 points, and forest, 9.7 points, while mountain is essentially matched at 0.2 points and glacier accounts for only 7.4. Glacier and mountain together make up just 18% of the total shortfall. Street and forest are texture-rich classes where ImageNet pretraining pays off most, which supports reading the gap as a difference in learned features rather than in raw capacity. Model B's accuracy is also bounded by the 12,000 training images in a way the pretrained models' is not, so more data would likely help it more than it would help them. Against that, it took a single training run of 20 epochs and no pretrained weights, it has no licensing or provenance constraints attached to ImageNet, and its behaviour is fully determined by code the group wrote and can audit, which matters for a deployed embedded system.

**Limitations of the SOTA route.** Beyond size, fine-tuning carried real complexity: both notebooks needed a two-phase schedule with the backbone frozen and then partially unfrozen, careful handling of BatchNorm in inference mode during fine-tuning, and undoing notebook 01's normalization to match each backbone's expected input range. Model B needed none of that. The pretrained models also inherit whatever biases and domain assumptions ImageNet carries.

**Recommendation.** Under a strict edge budget, Model B: it costs 0.3 MB, runs at 11.5M MACs, and gives up 7 points. Where several megabytes and roughly 30 to 50M MACs are available, MobileNetV2 is the better choice for accuracy, though its result here should be quoted with the 96x96 input stated. EfficientNet-B0 sits in between and is the better pick when compute rather than storage is the binding constraint. The fact that Model B also beats Model A, at one twenty-first the size, shows the budget was met by better architecture rather than by accepting a worse model.

## 11. Save Final Results

In [ ]:
print("Saved artifacts for the report:")
for path in (final_table_path, relative_path, tradeoff_path, efficiency_path, cm_grid_path, recall_path):
    print(f"  {path}  ({os.path.getsize(path) / 1024:.1f} KB)")

## 12. Overall Conclusion

In [ ]:
best = comparison_df.loc[comparison_df["Test Accuracy"].idxmax()]
smallest = comparison_df.loc[comparison_df["Total Parameters"].idxmin()]

print("EN3150 Assignment 03 - final results")
print("=" * 62)
print(f"Dataset:            {metrics_b['dataset_name']}, 6 classes, 64x64, 70/15/15 split")
print(f"Most accurate:      {best['Model']} at {best['Test Accuracy']:.4f} "
      f"({best['Total Parameters']:,} params)")
print(f"Smallest:           {smallest['Model']} at {smallest['Test Accuracy']:.4f} "
      f"({smallest['Total Parameters']:,} params, {smallest['Weights fp32 (KB)']:.0f} KB)")
print(f"Accuracy gap:       {(best['Test Accuracy'] - smallest['Test Accuracy']) * 100:.2f} percentage points")
print(f"Size ratio:         {best['Total Parameters'] / smallest['Total Parameters']:.1f}x")
print(f"Model B budget:     {metrics_b['trainable_params']:,} / 100,000 trainable parameters")

**Summary across the six tasks.**

1. **Data preparation.** Intel Image Classification, 17,034 labelled images across 6 scene classes, downscaled to 64x64 and split 70/15/15 with stratification at seed 42, giving 11,923 / 2,555 / 2,556 images. Every model in this notebook was trained and evaluated on those identical splits.
2. **Architecture design.** Model A, a standard CNN, reached 1,440,998 parameters. Model B replaced its convolutions with depthwise separable ones and its `Flatten -> Dense(256)` head with global average pooling, landing at 66,182 trainable parameters, comfortably inside the 100,000 budget and 21.4 times smaller than Model A.
3. **Optimizer selection.** Adam at a learning rate of 0.001 was selected on validation loss against plain SGD and SGD with momentum, with the test set held out of the selection.
4. **Custom model training and evaluation.** Both models trained for 20 epochs. Model B reached 83.18% test accuracy against Model A's 82.08%, so removing 95% of the parameters cost nothing and in fact gained 1.1 points. Model B also reached a lower best validation loss, 0.456 against 0.512. Their final train/validation accuracy gaps are similar, 0.125 and 0.112, so this is not a case of the smaller model simply overfitting less; capacity was not the binding constraint on this dataset, and the 1.37M extra parameters in Model A bought nothing.
5. **SOTA fine-tuning.** MobileNetV2 reached 90.38% at 96x96 input and EfficientNet-B0 89.24% at 64x64, both by two-phase fine-tuning from ImageNet weights.
6. **Final comparison.** The pretrained models buy 6 to 7 points of accuracy for 34 to 60 times the memory. Per megabyte of weights Model B is 30 to 50 times more efficient, and it is the only one of the four that would fit a microcontroller-class device.

**TODO before submission:** paste the final table and the two trade-off figures into the report, and state the GPU/CPU caveat from Section 9 wherever per-epoch training times are quoted.